# MyAI K5 — Data 100M in 2 halves (CPU-only)
v4 died at the 12h cell timeout inside 32k-BPE on 400M chars (pair-heap blowup). Fix: two 70k-doc halves; BPE trained on half A only (statistics saturate); both halves packed with A's tokenizer (identical vocab). Every cell <2h.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = secret_value_0
    print("HF_TOKEN attached")
except Exception as e:
    print(f"HF_TOKEN not attached ({type(e).__name__}) — continuing unauthenticated")
!git clone https://github.com/aloksingh9667/llm.git myai 2>&1 | tail -1
%cd /kaggle/working/myai
!pip install -q -r requirements/train.txt 2>&1 | tail -1

In [ ]:
!python scripts/stream_fineweb.py --skip-docs 0 --max-docs 70000 --max-tokens 50000000 --timeout-min 120 --out data/rawA/fineweb-A.txt

In [ ]:
!python scripts/prepare_tokenizer_corpus.py --raw-dir data/rawA --corpus data/tokenizer_corpus/train-A.txt --manifest data/tokenizer_corpus/manifest-A.json --name myai-corpus-part-a

In [ ]:
!python scripts/train_tokenizer.py --corpus data/tokenizer_corpus/train-A.txt --vocab-size 32000 --out data/tokenizer_corpus/tokenizer-32k.json --fast --sample-chars 15000000
!python scripts/eval_tokenizer.py --tokenizer data/tokenizer_corpus/tokenizer-32k.json --corpus data/tokenizer_corpus/train-A.txt --report /kaggle/working/tokenizer-32k-eval.json
!test -f data/tokenizer_corpus/tokenizer-32k.json && echo TOKENIZER_OK

In [ ]:
%%writefile configs/data-pack-fwA.yaml
version: "1.0.0"
corpus_path: data/tokenizer_corpus/train-A.txt
tokenizer_path: data/tokenizer_corpus/tokenizer-32k.json
output_dir: data/processed/fwA-32k
seq_len: 512
val_ratio: 0.01

In [ ]:
!python scripts/prepare_sequences.py --config configs/data-pack-fwA.yaml --corpus-manifest data/tokenizer_corpus/manifest-A.json --rows-per-shard 2048
!test -f data/processed/fwA-32k/manifest.json && echo PACK_A_OK

In [ ]:
!python scripts/stream_fineweb.py --skip-docs 70000 --max-docs 70000 --max-tokens 50000000 --timeout-min 120 --out data/rawB/fineweb-B.txt
!python scripts/prepare_tokenizer_corpus.py --raw-dir data/rawB --corpus data/tokenizer_corpus/train-B.txt --manifest data/tokenizer_corpus/manifest-B.json --name myai-corpus-part-b

In [ ]:
%%writefile configs/data-pack-fwB.yaml
version: "1.0.0"
corpus_path: data/tokenizer_corpus/train-B.txt
tokenizer_path: data/tokenizer_corpus/tokenizer-32k.json
output_dir: data/processed/fwB-32k
seq_len: 512
val_ratio: 0.01

In [ ]:
!python scripts/prepare_sequences.py --config configs/data-pack-fwB.yaml --corpus-manifest data/tokenizer_corpus/manifest-B.json --rows-per-shard 2048
!test -f data/processed/fwB-32k/manifest.json && echo PACK_B_OK

In [ ]:
!cp data/tokenizer_corpus/tokenizer-32k.json /kaggle/working/
!cp data/processed/fwA-32k/manifest.json /kaggle/working/packA-manifest.json
!cp data/processed/fwB-32k/manifest.json /kaggle/working/packB-manifest.json
!python -c "import json; [print(k, json.load(open(f'/kaggle/working/{k}-manifest.json'))['total_tokens'], 'tokens') for k in ('packA','packB')]"